In [36]:
import os
import re
from kiwipiepy import Kiwi
from dotenv import load_dotenv
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams

load_dotenv()

True

In [37]:
credentials = {
    "url": os.getenv("WATSONX_URL"),
    "apikey": os.getenv("WATSONX_APIKEY")
}
project_id = os.getenv("WATSONX_PROJECT_ID")

In [38]:
db_fetched_data = """
양치하기 싫다고 도망 다니더니 악어 칫솔 주니까 순순히 입을 벌림. 구석구석 깨끗하게 닦아주니 개운한지 거울 보고 이빨 자랑함. 스스로 치약 짜보겠다고 떼써서 살짝 진땀 뺌.
"""

In [ ]:
kiwi = Kiwi()

# 1. [NameError 해결 핵심] 문장 분리 처리 구역 (반드시 포함되어야 합니다)
refined_data = db_fetched_data.strip().replace('\n', ' ')
raw_lines = re.split(r'\.(?=\s|$)', refined_data)
lines = [line.strip() + "." for line in raw_lines if line.strip()]

# 2. 필수 부사 예외 매핑 사전
DICTIONARY_MAP = {
    "잘": "자다",
    "안": "않다",
    "못": "못하다"
}

step1_insights = ""

# 하드코딩 없는 100% 품사 추적 기반 사전형 변형 루프
for i, line in enumerate(lines):
    tokens = kiwi.tokenize(line)
    
    # [인덱스 꼬임 방지]: 원본 문자열을 직접 자르지 않고, 변환된 문자열 조각들을 모으는 독립 버퍼 기법 도입
    transformed_parts = []
    last_idx = 0
    
    j = 0
    while j < len(tokens):
        t = tokens[j]
        
        # 가. 동사(VV) 및 형용사(VA) 구역 발견 시 처리
        if t.tag in ["VV", "VA"]:
            # 이전 토큰의 끝 지점부터 현재 용언 어간 시작점까지의 원문 보존
            transformed_parts.append(line[last_idx:t.start])
            
            lemma = t.form if t.form.endswith("다") else f"{t.form}다"
            transformed_parts.append(lemma)
            
            # 연결된 어미(E로 시작하는 품사 태그: EC, EF, ETN 등)의 최종 범위를 한 번에 스캔하여 도려냄
            end_idx = t.end
            next_step = j + 1
            while next_step < len(tokens):
                if tokens[next_step].tag.startswith("E"):
                    end_idx = tokens[next_step].end
                    next_step += 1
                else:
                    break
            
            last_idx = end_idx
            j = next_step  # 스캔한 어미 토큰들을 건너뜀
            continue
            
        # 나. 필수 부사어 구역 발견 시 처리
        elif t.tag == "MAG" and t.form in DICTIONARY_MAP:
            transformed_parts.append(line[last_idx:t.start])
            transformed_parts.append(DICTIONARY_MAP[t.form])
            last_idx = t.end
            
        j += 1
        
    # 변환 후 남은 문장 뒷부분(명사, 조사 등)을 안전하게 결합
    transformed_parts.append(line[last_idx:])
    transformed_line = "".join(transformed_parts)

    # 다. 문장 부호 및 공백 정돈
    transformed_line = re.sub(r'\s+', ' ', transformed_line).strip()
    if not transformed_line.endswith("."):
        transformed_line += "."
    transformed_line = re.sub(r'\s+\.', '.', transformed_line)

    # 1단계 인사이츠에 깨끗한 형태로 적재
    step1_insights += f"{i+1}. {transformed_line}\n"

step1_insights = step1_insights.strip()
print(f"-> 2단계 프롬프트로 주입될 최종 step1_insights 데이터:\n{step1_insights}")


-> 2단계 프롬프트로 주입될 최종 step1_insights 데이터:
1. 양치하기 싫다 도망 다니다 악어 칫솔 주니까 순순히 입을 벌리다.
2. 구석구석 깨끗하게 닦다주니 개운한지 거울 보다 이빨 자랑함.
3. 스스로 치약 짜다보겠다고 떼쓰다 살짝 진땀 빼다.


In [40]:
refined_data = db_fetched_data.strip().replace('\n', ' ')
raw_lines = re.split(r'\.(?=\s|$)', refined_data)
lines = [line.strip() + "." for line in raw_lines if line.strip()]

In [41]:
extract_params = {
    GenParams.DECODING_METHOD: "greedy",
    GenParams.MIN_NEW_TOKENS: 1,
    GenParams.MAX_NEW_TOKENS: 1000,
    GenParams.REPETITION_PENALTY: 1.2,
}

extractor_model = ModelInference(
    model_id="mistralai/mistral-small-3-1-24b-instruct-2503",
    credentials=credentials,
    params=extract_params,
    project_id=project_id
)

In [42]:
extract_prompt = f"""[Instruction]
당신은 육아 기록 전문가입니다. 
주어진 [Data]의 각 문장을 순서대로 정밀 분석하여 아래 [Output Format] 양식에 맞춰 오직 핵심 라벨 결과만 깨끗하게 출력하세요. 
원문의 글자 형태를 절대로 임의로 변형하거나 깨뜨리지 마십시오.
핵심어
행동(무엇을 하다)·대상(무엇을)·장소(어디서)·시간(언제)·이유(왜)·상태(어떠한가)·방식(어떻게)·결과(어떻게 되다)·수치(얼마나)·상대(누구와)·감정(어떤 기분으로)·대책(어떻게 대처했나) 
중에서 문장에 존재하는 것들만 사전에 적혀있는 형태로 출력하라.

감정
기쁨, 행복, 설렘, 감사, 만족, 편안, 신남, 뿌듯함, 성취감, 안도, 다행, 환희, 유쾌, 상쾌, 평온, 든든함, 활력, 희열, 감격, 낙천, 재미, 황홀, 흡족, 평화, 아늑함, 활기, 홀가분함, 경쾌함, 흐뭇함, 희망차다, 의연함, 쾌활함, 충만함, 짜릿함, 안락함, 가뿐함, 낙관, 쾌감, 환호, 통쾌, 즐거움, 흥겨움, 감동, 뭉클함, 벅차오름, 고무됨, 황홀경, 흔쾌함, 흡족함, 희망, 자신감, 충만, 평정, 고요, 온화, 안온.
슬픔, 무기력, 외로움, 서운함, 서글픔, 안쓰러움, 미안함, 허탈함, 우울, 낙담, 절망, 괴로움, 공허, 쓸쓸함, 상실감, 비통, 처량, 애통, 침울, 허무, 자책, 후회, 짠함, 비장함, 울적함, 참담함, 고독, 애잔함, 서량함, 가슴 아픔, 낙망, 비련, 먹먹함, 처절함, 비련, 비장미, 비참, 서글퍼짐, 가련함, 서글픔, 애련, 애수, 비탄, 한탄, 탄식, 수심, 시름, 낙심, 적적함, 고적함, 비량, 슬퍼함, 서글퍼함.
공포, 불안, 조마조마함, 두려움, 무서움, 긴장, 경계, 초조, 안절부절, 겁남, 공황, 섬뜩함, 소름, 위축, 당혹, 어리둥절, 당황, 의심, 의아함, 혼란, 막막함, 낯가림, 전전긍긍, 한기, 경악, 혼비백산, 삭막함, 아찔함, 아연실색, 전율, 주눅 듦, 소외감, 패닉, 의구심, 전전반측, 고심, 전전긍긍, 한심함, 근심, 걱정, 염려, 불안감, 공포감, 전율, 소름끼침, 소스라침, 전전, 초조함.
부끄러움, 죄책감, 자부심, 동정심, 호기심, 부러움, 수치심, 창피, 시기, 동경, 연민, 애정, 친밀, 자괴감, 열등감, 우월감, 자만, 기특함, 신기함, 경이로움, 대견함, 자랑스러움, 숭배, 존경, 신뢰, 냉소, 방관, 겸손, 거만, 열정, 헌신, 자비, 은혜, 수줍음, 계면쩍음, 미안함, 송구함, 민망함, 무안함, 면구스러움, 자긍심, 열등감, 시기심, 질투심, 시샘, 영광, 자만심, 오만, 거만함, 경외감, 신비롭다, 기이함.
귀여움, 엉뚱함, 당혹감, 웃김, 사랑스러움, 대단함, 팽팽함, 허탈감, 지루함, 졸림, 배고픔, 노곤함, 나른함, 개운함, 찝찝함, 아쉬움, 시원섭섭함, 섭섭함, 어색함, 황당함, 어이없음, 멍함, 권태, 심심함, 감흥 없음, 무덤덤함, 덤덤함, 시큰둥함, 냉담, 무관심, 냉정, 무정, 초연함, 의아함, 아리송함, 알쏭달쏭, 기가 막힘, 기절초풍, 질색, 따분함, 나태, 무료함.

이 중에서 출력하라.

육아범주는 수면, 식사, 사회성, 운동, 언어, 배변, 정서, 인지, 건강, 옷 이것들 중에서 출력하라.

핵심어, 감정, 육아범주 모두 하나 이상 채워져 있어야 한다.

[Data]
{step1_insights}

[Output Format]
1. 문장원문: [문장 내용]
- 핵심어: 단어1, 단어2
- 감정: 슬픔, 기쁨
- 육아범주: 수면

[Output]
"""


In [43]:
try:
    # [핵심 보완 1]: Kiwi 분석 결과가 담긴 step1_insights 변수를 프롬프트 템플릿에 명확히 주입하여 완성합니다.
    final_extract_prompt = extract_prompt.format(step1_insights=step1_insights)
    
    # [핵심 보완 2]: 원본 템플릿 대신, 데이터가 완벽히 채워진 final_extract_prompt를 모델에 넘깁니다.
    extract_response = extractor_model.generate(prompt=final_extract_prompt)
    
    if 'results' in extract_response and len(extract_response['results']) > 0:
        step2_keywords = extract_response['results'][0].get('generated_text', '').strip()
    else:
        step2_keywords = str(extract_response).strip()
except Exception as e:
    # 2단계(라벨 추출) 파이프라인이므로 로그 메시지만 명확하게 보정합니다.
    print(f"2단계 실행 중 오류 발생: {e}")
    step2_keywords = ""

perfect_match_input = step2_keywords


In [44]:
creative_params = {
    GenParams.DECODING_METHOD: "sample",  # 일기 생성 등 창의적 맥락에는 sample 방식이 자연스럽습니다.
    GenParams.MIN_NEW_TOKENS: 50,
    GenParams.MAX_NEW_TOKENS: 600,
    GenParams.REPETITION_PENALTY: 1.1,
    GenParams.TEMPERATURE: 0.1,
    GenParams.TOP_P: 0.8,
    GenParams.STOP_SEQUENCES: ["\n\n", "[END]"]
}

writer_model = ModelInference(
    model_id="meta-llama/llama-3-3-70b-instruct",
    credentials=credentials,
    params=creative_params,
    project_id=project_id
)

In [45]:
diary_prompt = f"""너는 인스타그램에서 오늘 하루의 기록을 다정하고 솔직하게 독백 형태로 공유하는 대한민국 엄마이다.
제공된 [육아 데이터 블록]의 '문장원문' 상황을 베이스로 삼고, 여기에 명시된 '핵심어', '감정', '육아범주' 라벨 정보들을 완벽하게 결합하여 한 번호당 정확히 한 문장씩 자연스러운 한국어 일기를 작성해라.

[출력 예시 - 이 자연스러운 문장 구조와 어투만 참고하여 라벨 정보를 문장으로 만드세요]
출근길에 지하철을 바로 타서 지각하지 않고 제시간에 안전하게 도착했네요.
칭찬받으려고 열심히 준비한 기획안을 부장님이 보시고 활짝 웃어주셔서 정말 뿌듯했답니다.
퇴근하고 집으로 돌아와 따뜻한 물로 샤워를 하니 하루의 피로가 싹 풀리더라고요.
산책하러 나가서 잔디밭을 신나게 뛰놀고 집으로 얌전하게 돌아왔네요.
기특해서 털에 묻은 먼지를 털어주고 부드럽게 쓰다듬어 주었답니다.
고맙다는 듯이 꼬리를 살랑살랑 흔들며 안기는데 하루의 스트레스가 싹 풀리더라고요.

[작성 규칙 - 절대 준수]
1. 문장 개수 1:1 일치: [육아 데이터 블록]의 번호 개수와 똑같은 개수의 문장만 작성해라. 오직 제공된 '문장원문'의 현실적인 상황 맥락만을 충실히 바탕으로 작성해라. 한 문장이 끝날 때마다 무조건 줄바꿈을 해라.
2. 자연스러운 감정 및 시제 반영: 
   - '문장원문'에 나타난 행동 묘사 속에 2단계 '감정' 라벨 단어의 정서가 인스타 감성으로 예쁘게 묻어나도록 자연스럽게 녹여내라. 오늘 실제로 겪은 과거의 일을 회상하듯 작성해라.
   - 데이터의 핵심어를 문맥에 맞게 서술형으로 작성해라.
3. 주어 및 목적어: 문장 시작할 때 상투적이지 않고 신선한 주어나 목적어를 써라. 핵심어 단어 원형에 조사와 어미를 자연스럽게 융합하여, 생생한 행동이나 상황 묘사로 문장을 곧바로 시작해라.
4. 어미 결합 규칙 (필수 준수): 문장의 마지막 어미는 반드시 과거형 받침 뒤에 자연스러운 대화형태로 끝마쳐라. 국립국어원 맞춤법과 띄어쓰기를 완벽하게 준수해라.
5. 깨끗한 한글 출력: 문장은 오직 순수한 한글로만 작성해야 하며, 외국어의 경우 오직 제공된 라벨에 명시되어 있는 경우에만 제한적으로 포함하여 출력해라.
6. 마감 기호: 모든 문장 작성을 마친 바로 다음 줄에 무조건 [END] 라고만 출력해라.

[육아 데이터 블록]
{perfect_match_input}

[Diary]:"""


In [46]:
# [핵심 보완]: 2단계에서 완벽하게 추출된 perfect_match_input 데이터를 diary_prompt 템플릿에 주입합니다.
final_diary_prompt = diary_prompt.format(perfect_match_input=perfect_match_input)

# 완성된 최종 프롬프트를 일기 생성 모델(writer_model)에 넘겨 가동합니다.
writer_response = writer_model.generate(prompt=final_diary_prompt)

writer_results = writer_response.get('results', [])
first_writer_result = next(iter(writer_results)) if isinstance(writer_results, list) and writer_results else {}
raw_diary = first_writer_result.get('generated_text', '').strip() if isinstance(first_writer_result, dict) else str(first_writer_result).strip()


In [47]:
if "[END]" in raw_diary:
    raw_diary = raw_diary.split("[END]")[0].strip()

raw_lines = [line.strip() for line in raw_diary.split('\n') if line.strip()]

full_print_lines = []
for line in raw_lines:
    line = re.sub(r'^\d+[\.\s\-~)]+|^\s*\[\d+[^\]]*\]', '', line).strip()
    
    line = re.sub(r'[\u4e00-\u9fff]', '', line)
    line = re.sub(r'[^가-힣a-zA-Z0-9\s\.,!\?\'\"~%·]', '', line).strip()
    
    if line:
        full_print_lines.append(line)

def truncate_by_bytes(text, max_bytes=400):
    text_bytes = text.encode('utf-8')
    if len(text_bytes) <= max_bytes:
        return text
    return text_bytes[:max_bytes - 3].decode('utf-8', errors='ignore').strip() + "..."

final_lines = []
for line in full_print_lines:
    final_lines.append(truncate_by_bytes(line, 400))

final_diary = "\n".join(final_lines)


In [48]:
print("\n=== 1단계: 구조화된 요약 메모 추출 완료 ===")
print(step1_insights)


=== 1단계: 구조화된 요약 메모 추출 완료 ===
1. 양치하기 싫다 도망 다니다 악어 칫솔 주니까 순순히 입을 벌리다.
2. 구석구석 깨끗하게 닦다주니 개운한지 거울 보다 이빨 자랑함.
3. 스스로 치약 짜다보겠다고 떼쓰다 살짝 진땀 빼다.


In [49]:
print("\n=== 2단계: 주요 라벨 단어 추출 완료 ===")
print(step2_keywords)


=== 2단계: 주요 라벨 단어 추출 완료 ===
1. 문장원문: 양치하기 싫다 도망 다니다 악어 칫솔 주니까 순순히 입을 벌리다.
   - 핵심어: 행동(양치하다), 대상(입), 이유(싫다)
   - 감정:
   - 육아범주: 건강

2. 문장원문: 구석구석 깨끗하게 닦다주니 개운한지 거울 보다 이빨 자랑함.
    - 핵심어: 행동(닦다), 대상(이빨), 장소(구석구석), 상태(깨끗하다), 방법(개운하다), 상대(아이)
    - 감정: 기쁨
    - 육아범주: 건강

3. 문장원문: 스스로 치약을 짜다 보려고 떼쓸다 살짝 진땀 뺐다.
     - 핵심어: 행동(짜다), 대상(치약), 방식(떼쓰다)
     - 감정: 부끄럼
     - 육아범주: 건강


In [50]:
print("\n=== 3단계: 최종 완성된 감성 일기 ===")
for idx, final_line in enumerate(final_lines):
    print(f"[{idx+1}번 일기]: {final_line}")
print(f"\n-> 최종 결과물 총 문장 수: {len(final_lines)}줄")


=== 3단계: 최종 완성된 감성 일기 ===
[1번 일기]: 양치를 해야 하는데도 불구하고 양치하기를 싫어하는 우리 아이가 도망을 다녀서, 악어 모양의 칫솔을 보여주니 순순히 입을 벌려주었어요.
[2번 일기]: 구석구석을 깨끗하게 닦아줘서 개운한지, 거울을 보더니 이빨을 자랑하며 너무 기뻐했답니다.
[3번 일기]: 스스로 치약을 짜보려고 떼를 쓰더니, 살짝 진땀을 빼며 성공했다는 표정을 지었지만, 그 모습이 너무 부끄러워 보였답니다.

-> 최종 결과물 총 문장 수: 3줄
